# AlphaTrain Pillar 2k: Survival Clock (Hybrid Life Energy)

Root cause found: TD returns have ZERO correlation with board health (r=-0.036).
Value head can't distinguish expert moves from random moves (43.8 vs 43.2).
Dying boards get 3x overestimated (predicted 23 vs truth 7.9).

**Fix: Switch value target to survival + scoring hybrid.**
- Per-turn reward: `r(t) = 1.0 + points_scored / 10`
- Discounted returns with γ=0.95 (half-life 14 turns)
- Death → V≈1, Healthy → V≈25, Scoring streak → V≈35
- 128 bins over [0, 30], resolution 0.24 pts/bin

**Also keeping from 2k-alpha:**
- Bigger value head (32ch conv, 512 hidden, dropout 0.3)
- Adversarial ranking (top-1 vs random move)
- 30% endgame oversampling

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_selfplay_train.tar.gz` — code
2. `expert_v2_pairwise_g095_surv1.0_ms30.pt.gz` — survival hybrid data
3. `pillar2k_best.pt` — base model (warm start, only value_fc2 reinits)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time

DRIVE = '/content/drive/MyDrive/alphatrain'

# Extract code (always fresh)
!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

# Copy model (always fresh)
os.makedirs('/content/alphatrain/data', exist_ok=True)
for f in ['pillar2k_best.pt']:
    src = os.path.join(DRIVE, f)
    dst = os.path.join('/content/alphatrain/data', f)
    print(f'Copying {f}...')
    shutil.copy(src, dst)
    print(f'{f}: {os.path.getsize(dst)/1e6:.0f} MB')

# Decompress survival hybrid data (always fresh)
gz_src = os.path.join(DRIVE, 'expert_v2_pairwise_g095_surv1.0_ms30.pt.gz')
pt_dst = '/content/alphatrain/data/expert_v2_pairwise_g095_surv1.0_ms30.pt'
print('Decompressing survival hybrid data...')
t0 = time.time()
!gzip -dc {gz_src} > {pt_dst}
print(f'Done in {time.time()-t0:.0f}s: {os.path.getsize(pt_dst)/1e9:.1f} GB')

!pip install -q numpy numba scipy pytest

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

# Verify model architecture (128 bins)
from alphatrain.model import AlphaTrainNet
m = AlphaTrainNet(value_channels=32, value_hidden=512, value_dropout=0.3, num_value_bins=128)
n = sum(p.numel() for p in m.parameters())
nv = sum(p.numel() for n, p in m.named_parameters() if 'value' in n)
print(f'\nModel: {n:,} params (value head: {nv:,})')
del m

# Verify data
d = torch.load(pt_dst, weights_only=True, map_location='cpu')
print(f"\nData: {d['boards'].shape[0]:,} states")
print(f"  gamma={d['gamma']}, max_score={d['max_score']}, bins={d['num_value_bins']}")
print(f"  turns_remaining: {'turns_remaining' in d}")
if 'turns_remaining' in d:
    tr = d['turns_remaining']
    print(f"  endgame (<=100 turns): {(tr <= 100).sum().item():,}")
del d

!cd /content && python -m pytest alphatrain/tests/test_model.py -v --tb=short 2>&1 | tail -5

In [ ]:
# Pillar 2k: Survival Clock (Hybrid Life Energy)
# r(t) = 1.0 + points/10, gamma=0.95, 128 bins, max_score=30
# Bigger value head (32ch/512h), dropout 0.3, adversarial ranking
# Warm start from 2k-alpha (only value_fc2 reinits: 64->128 bins)
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train \
    --tensor-file alphatrain/data/expert_v2_pairwise_g095_surv1.0_ms30.pt \
    --gpu-data --amp --compile \
    --value-bins 128 --value-channels 32 --value-hidden 512 --value-dropout 0.3 \
    --val-weight 0.01 --rank-weight 1.0 \
    --endgame-fraction 0.3 --endgame-threshold 100 \
    --adversarial-ranking \
    --resume alphatrain/data/pillar2k_best.pt --warm-start \
    --epochs 10 --batch-size 8192 --lr 1e-4 --warmup-epochs 2 \
    --copy-to /content/drive/MyDrive/alphatrain/pillar2k_surv_best.pt \
    --save-dir /content/checkpoints/pillar2k_surv

In [ ]:
# Evaluate policy score
%cd /content
!python -m alphatrain.evaluate --player policy \
    --model /content/checkpoints/pillar2k_surv/best.pt \
    --games 50 --seed 42

In [ ]:
# Copy final model to Drive
import shutil, os
DRIVE = '/content/drive/MyDrive/alphatrain'
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/pillar2k_surv/{f}'
    dst = f'{DRIVE}/pillar2k_surv_{f}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')